# Athena emulator — quickstart

You've just brought up the local Athena emulator stack and want your first
query end to end. In a handful of cells you'll connect with boto3, run a
`SELECT`, see where the result lands on S3, and read it back as a DataFrame
with awswrangler.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) ·
`trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).
Per ADR-0002, only Athena traffic goes to the emulator; S3/Glue calls must
still land on moto.

In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid

import boto3
from _evidence import persist, record, render

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

## Where your clients land

`client.meta.endpoint_url` reports where each boto3 client actually lands.
Per ADR-0002 only Athena traffic goes to the emulator (`:5001`); S3/Glue
stay on moto (`:5000`).

In [3]:
assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL, (
    athena.meta.endpoint_url
)
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL, s3.meta.endpoint_url
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL, glue.meta.endpoint_url

record(
    "endpoint routing (athena→emulator, s3/glue→moto)",
    "PASS",
    f"athena={athena.meta.endpoint_url} s3={s3.meta.endpoint_url} glue={glue.meta.endpoint_url}",
)

Evidence(feature='endpoint routing (athena→emulator, s3/glue→moto)', status='PASS', detail='athena=http://localhost:5001 s3=http://localhost:5000 glue=http://localhost:5000', latency_ms=None)

## Your first query

Throwaway bucket + Glue database per run — the stack's data is user data,
never reset. `start_query_execution` → poll `get_query_execution` →
`get_query_results`, measuring start→SUCCEEDED latency.

In [4]:
bucket = f"nb01-{uuid.uuid4().hex[:12]}"
database = f"nb01_{uuid.uuid4().hex[:8]}"
s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})
s3_output = f"s3://{bucket}/results/"

started = time.monotonic()
query_id = athena.start_query_execution(
    QueryString="SELECT 1 AS one, 'alpha' AS label",
    QueryExecutionContext={"Database": database},
    ResultConfiguration={"OutputLocation": s3_output},
    WorkGroup="primary",
)["QueryExecutionId"]

state = "QUEUED"
while state in {"QUEUED", "RUNNING"}:
    time.sleep(0.5)
    state = athena.get_query_execution(QueryExecutionId=query_id)[
        "QueryExecution"
    ]["Status"]["State"]
elapsed_ms = (time.monotonic() - started) * 1000

assert state == "SUCCEEDED", f"{state} on {query_id}"
result_rows = athena.get_query_results(QueryExecutionId=query_id)["ResultSet"][
    "Rows"
]
assert result_rows[0]["Data"][0]["VarCharValue"] == "one"  # header row
assert result_rows[1]["Data"][0]["VarCharValue"] == "1"
assert result_rows[1]["Data"][1]["VarCharValue"] == "alpha"

record(
    "boto3 start→poll→get_query_results",
    "PASS",
    f"{query_id} SUCCEEDED, {len(result_rows) - 1} data row(s)",
    latency_ms=elapsed_ms,
)
print(f"{query_id} → {state} in {elapsed_ms:.0f} ms")

86660956-1714-4842-a17b-37b214284f83 → SUCCEEDED in 515 ms


## Where the result landed

A DML `SELECT` writes `{QueryID}.csv` (quoted header row as line 1) plus
`{QueryID}.csv.metadata` under `OutputLocation` before the execution reports
SUCCEEDED (ADR-0007, ADR-0010).

In [5]:
objects = s3.list_objects_v2(Bucket=bucket, Prefix="results/")["Contents"]
keys = sorted(obj["Key"] for obj in objects)
print("\n".join(keys))

assert f"results/{query_id}.csv" in keys
assert f"results/{query_id}.csv.metadata" in keys

body = (
    s3.get_object(Bucket=bucket, Key=f"results/{query_id}.csv")["Body"]
    .read()
    .decode("utf-8")
)
assert body.splitlines()[0] == '"one","label"'
assert body.splitlines()[1] == '"1","alpha"'

record(
    "result artifacts on moto S3 (.csv + .csv.metadata)",
    "PASS",
    "; ".join(keys),
)

results/86660956-1714-4842-a17b-37b214284f83.csv
results/86660956-1714-4842-a17b-37b214284f83.csv.metadata


Evidence(feature='result artifacts on moto S3 (.csv + .csv.metadata)', status='PASS', detail='results/86660956-1714-4842-a17b-37b214284f83.csv; results/86660956-1714-4842-a17b-37b214284f83.csv.metadata', latency_ms=None)

## Read it as a DataFrame

One `wr.athena.read_sql_query` call (csv path, non-managed `primary`
workgroup) reads the result straight into pandas.

In [6]:
import awswrangler as wr

frame = wr.athena.read_sql_query(
    sql="SELECT 1 AS one, 'alpha' AS label",
    database=database,
    ctas_approach=False,
    s3_output=s3_output,
)
assert list(frame.columns) == ["one", "label"]
assert frame.loc[0, "one"] == 1
assert frame.loc[0, "label"] == "alpha"

record(
    "wr.athena.read_sql_query (csv path)",
    "PASS",
    f"shape={frame.shape} qid={frame.query_metadata['QueryExecutionId']}",
)
frame

,one,label
0,1,alpha


## Capabilities demonstrated

Every feature above recorded a measured row. The matrix persists to
`parity/01_quickstart.md`; `PARITY.md` at the project root is regenerated
from all fragments.

In [7]:
print(render())
persist("01_quickstart")

| feature | status | detail | latency_ms |
|---|---|---|---|
| endpoint routing (athena→emulator, s3/glue→moto) | PASS | athena=http://localhost:5001 s3=http://localhost:5000 glue=http://localhost:5000 |  |
| boto3 start→poll→get_query_results | PASS | 86660956-1714-4842-a17b-37b214284f83 SUCCEEDED, 1 data row(s) | 515.157 |
| result artifacts on moto S3 (.csv + .csv.metadata) | PASS | results/86660956-1714-4842-a17b-37b214284f83.csv; results/86660956-1714-4842-a17b-37b214284f83.csv.metadata |  |
| wr.athena.read_sql_query (csv path) | PASS | shape=(1, 2) qid=eb1b0173-2d8a-46f3-ad72-f78194f5388b |  |



PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/01_quickstart.md')

## Clean up

The run's bucket and database are throwaway — delete them so the stack is
left as you found it.

In [8]:
contents = s3.list_objects_v2(Bucket=bucket).get("Contents", [])
if contents:
    s3.delete_objects(
        Bucket=bucket,
        Delete={"Objects": [{"Key": obj["Key"]} for obj in contents]},
    )
s3.delete_bucket(Bucket=bucket)
glue.delete_database(Name=database)
print(f"cleaned bucket={bucket} database={database}")

cleaned bucket=nb01-ee882cfe5cb8 database=nb01_5d80a9bf
